# MediCS — Colab GPU Runner

Run cells in order. Each cell is independent — re-run any cell to check scores.

**Pipeline:** Setup → Dataset → Attack-Defense Loop (3 rounds) → DPO → Eval → Transfer → Detection → Figures → Upload

In [5]:
# Cell 1 — Setup (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import sys, os
os.chdir('/content/drive/MyDrive/MediCS')
sys.path.insert(0, '/content/drive/MyDrive/MediCS')

%pip install -q -r requirements.txt
!nvidia-smi
print(f"\nWorking dir: {os.getcwd()}")
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sun Mar 29 22:08:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|          

## Phase A — Dataset Construction (one-time, no GPU needed)

In [2]:
# Cell 2 — Build MediCS-500 dataset (seeds → keywords → translation → splits)
# NOTE: Calls OpenAI API for keyword extraction — costs ~$1-2
!python scripts/01_build_dataset.py --config config/experiment_config.yaml

[TIMING] Starting: Dataset Construction
API keys loaded from .env file.
=== MediCS-500 Dataset Builder ===
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
Semantic threshold: 0.75

--- Step 1-2: Loading and deduplicating seeds ---
Loaded 500 raw seeds
Deduplication: 500 -> 500 (removed 0 duplicates)
After dedup: 500 seeds → saved to data/seeds/deduped_seeds.jsonl

--- Step 3: Extracting keywords ---
  Resuming from checkpoint: 500 keywords already extracted
  All 500 keywords already extracted (from checkpoint)
Extracted keywords for 500 seeds

--- Step 4: Code-switching ---
Code-switching: 100% 3000/3000 [00:03<00:00, 849.69it/s] 
Generated 3000 code-switched variants

=== Semantic Preservation Verification ===
modules.json: 100% 349/349 [00:00<00:00, 1.31MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 437kB/s]
README.md: 10.5kB [00:00, 4.96MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 277kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.35MB/s]
model

In [23]:
import os
from huggingface_hub import login, whoami

# 1) Remove bad env token from current session
os.environ.pop("HF_TOKEN", None)

# 2) Paste a NEW valid personal token
new_tok = "YOUR_HF_TOKEN_HERE"   # new token from your HF account
login(token=new_tok, add_to_git_credential=False)

# 3) Verify
print(whoami(token=new_tok)["name"])


DarthVaderr


In [34]:
import os
from pathlib import Path
from medics.utils import load_dotenv
from huggingface_hub import whoami

os.chdir('/content/drive/MyDrive/MediCS')
print("cwd:", os.getcwd(), "env_exists:", Path(".env").exists())

# Clear stale token from current runtime, then force reload from .env
os.environ.pop("HF_TOKEN", None)
load_dotenv(override=True)

tok = os.environ.get("HF_TOKEN", "")
print("HF_TOKEN loaded:", bool(tok), "len:", len(tok), "prefix_ok:", tok.startswith("hf_"))

print("HF user:", whoami(token=tok)["name"])


cwd: /content/drive/MyDrive/MediCS env_exists: True
HF_TOKEN loaded: True len: 37 prefix_ok: True
HF user: DarthVaderr


In [35]:
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    filename="config.json",
    token=tok
)
print("Llama access OK")


Llama access OK


In [ ]:
# Cell 3 — Token fragmentation analysis (no GPU, no API cost)
!python scripts/07_tokenization_analysis.py

[TIMING] Starting: Tokenization Analysis
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loaded 2665 seeds from data/medics_500/medics_500_full.jsonl
Loaded keywords for 500 seeds

Analyzing tokenization for meta-llama/Meta-Llama-3-8B-Instruct
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
  Tokenization analysis: 50/2665 seeds processed


## Round 1 — Base Model Attack

In [ ]:
# Cell 4 — Round 1: Generate attack prompts (Thompson Sampling)
!python scripts/02_run_attack_round.py --round 1 --phase generate

In [ ]:
# Cell 5 — Round 1: Inference (base model, BASE prompt)
!python colab/run_inference.py \
    --checkpoint base --system-prompt base --seed 42 \
    --input results/attacks/round_1/prompts.jsonl \
    --output results/attacks/round_1/responses.jsonl

In [ ]:
# Cell 6 — Round 1: Judge responses (GPT-5)
# NOTE: This calls the OpenAI API — costs ~$0.40
!python scripts/02_run_attack_round.py --round 1 --phase judge

In [ ]:
# Cell 7 — Round 1: Check ASR
from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_1/results.jsonl")
print(f"Round 1 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 8 — Round 1: Build SFT data + Train SFT + Benign eval for DPO
!python scripts/03_build_defense_data.py --rounds 1 --type sft
!python colab/train_sft.py --round 1

# Benign inference through SFT round 1 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_1/benign_results.jsonl

## Round 2 — Attack SFT Round 1

In [ ]:
# Cell 9 — Round 2: Generate attacks
!python scripts/02_run_attack_round.py --round 2 --phase generate

In [ ]:
# Cell 10 — Round 2: Inference (SFT round 1 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input results/attacks/round_2/prompts.jsonl \
    --output results/attacks/round_2/responses.jsonl

In [ ]:
# Cell 11 — Round 2: Judge + ASR
!python scripts/02_run_attack_round.py --round 2 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_2/results.jsonl")
print(f"\nRound 2 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 12 — Round 2: Build SFT data + Train SFT + Benign eval for DPO
!python scripts/03_build_defense_data.py --rounds 1,2 --type sft
!python colab/train_sft.py --round 2 --prev-checkpoint checkpoints/sft/round_1/final

# Benign inference through SFT round 2 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_2/benign_results.jsonl

## Round 3 — Attack SFT Round 2

In [ ]:
# Cell 13 — Round 3: Generate attacks
!python scripts/02_run_attack_round.py --round 3 --phase generate

In [ ]:
# Cell 14 — Round 3: Inference (SFT round 2 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input results/attacks/round_3/prompts.jsonl \
    --output results/attacks/round_3/responses.jsonl

In [ ]:
# Cell 15 — Round 3: Judge + ASR
!python scripts/02_run_attack_round.py --round 3 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_3/results.jsonl")
print(f"\nRound 3 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 16 — Final SFT (round 3) + Benign eval + DPO
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type sft
!python colab/train_sft.py --round 3 --prev-checkpoint checkpoints/sft/round_2/final

# Benign inference through SFT round 3 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_3/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_3/benign_results.jsonl

# Build DPO data (uses benign results from all 3 SFT rounds for over-refusal correction)
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type dpo
!python colab/train_dpo.py --sft-checkpoint checkpoints/sft/round_3/final

## Held-Out Evaluation — 3 Checkpoints x 3 Seeds

In [ ]:
# Cell 17 — Held-out evaluation: all checkpoints x all seeds (BASE prompt throughout)
checkpoints = {
    "base": "base",
    "sft": "checkpoints/sft/round_3/final",
    "dpo": "checkpoints/dpo/final",
}
seeds = [42, 123, 456]

for ckpt_name, ckpt_path in checkpoints.items():
    for seed in seeds:
        # Attack prompts
        out = f"results/eval/{ckpt_name}/seed_{seed}/held_out.jsonl"
        print(f"\n{'='*60}")
        print(f"Running: {ckpt_name} seed={seed} (attacks)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/splits/held_out_eval.jsonl \
            --output {out}

        # Benign twins (for helpfulness judging)
        benign_out = f"results/eval/{ckpt_name}/seed_{seed}/benign_results.jsonl"
        print(f"Running: {ckpt_name} seed={seed} (benign)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/seeds/benign_twins.jsonl \
            --output {benign_out}

In [ ]:
# Cell 18 — Helpfulness judging (must run BEFORE metric evaluation)
# Judge benign results for all checkpoints x all seeds
import itertools
checkpoints = ["base", "sft", "dpo"]
seeds = [42, 123, 456]

for ckpt, seed in itertools.product(checkpoints, seeds):
    inp = f"results/eval/{ckpt}/seed_{seed}/benign_results.jsonl"
    print(f"Judging helpfulness: {ckpt} seed={seed}")
    !python scripts/04_evaluate.py --judge-helpfulness --input {inp}

In [ ]:
# Cell 19 — Full metric evaluation (ASR, HR, FRR, McNemar, Cohen's h)
# Run AFTER helpfulness judging so HR/FRR are accurate
!python scripts/04_evaluate.py --checkpoints base,sft,dpo --seeds 42,123,456

## Transfer + Detection + Figures + Upload

In [ ]:
# Cell 20 — Transfer evaluation (Mistral-7B, BASE prompt)
!python colab/run_transfer.py --input data/splits/held_out_eval.jsonl --seed 42
!python scripts/04_evaluate.py --judge-transfer --input results/transfer/mistral_results.jsonl

In [ ]:
# Cell 21 — Perplexity detection baseline
!python colab/run_perplexity.py --checkpoint base \
    --input data/splits/held_out_eval.jsonl \
    --english-input data/seeds/raw_seeds.jsonl \
    --output results/analysis/perplexity_results.json
!python scripts/08_detection_analysis.py

In [ ]:
# Cell 22 — Residual analysis + Figures + Cost report
!python scripts/04_evaluate.py --residual-analysis --checkpoint dpo
!python scripts/05_generate_figures.py --results-dir results/eval/
!python scripts/09_cost_report.py

In [ ]:
# Cell 23 — Upload to HuggingFace Hub
!python scripts/06_upload_hf.py

## ASR Dashboard — Re-run anytime to check all scores

In [ ]:
# Cell 24 — ASR Dashboard: all rounds at a glance
from medics.metrics import compute_asr, compute_per_strategy_asr
from medics.utils import load_jsonl
from pathlib import Path

print("=" * 60)
print("  MediCS ASR Dashboard")
print("=" * 60)

# Attack rounds
for r in range(1, 4):
    path = f"results/attacks/round_{r}/results.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        n_harmful = sum(1 for x in results if x.get("judge_label") == "harmful")
        strats = compute_per_strategy_asr(results)
        strat_str = " | ".join(f"{s}:{v:.0%}" for s, v in sorted(strats.items()))
        print(f"\n  Round {r}: ASR {asr:.1%} ({n_harmful}/{len(results)})")
        print(f"    {strat_str}")
    else:
        print(f"\n  Round {r}: not yet run")

# Held-out evaluation
print(f"\n{'='*60}")
print("  Held-Out Evaluation (seed=42)")
print("=" * 60)
for ckpt in ["base", "sft", "dpo"]:
    path = f"results/eval/{ckpt}/seed_42/held_out.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        print(f"  {ckpt:>5}: ASR {asr:.1%}")
    else:
        print(f"  {ckpt:>5}: not yet run")

# Transfer
transfer_path = "results/transfer/mistral_results.jsonl"
if Path(transfer_path).exists():
    results = load_jsonl(transfer_path)
    asr = compute_asr(results)
    print(f"\n  Transfer (Mistral-7B): ASR {asr:.1%}")